In [1]:
from gymnasium import Env
from gymnasium.spaces import Discrete, Box, Dict
import numpy as np
import random
from typing import Optional
import pygame
import os

C:\Users\dogem\miniforge3\envs\ML_ENV\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
path = os.path.join("../", "data", "data3.npz") 
path2 = os.path.join("../", "data", "traindata.npz") 

master = np.load(path)
small_master = np.load(path2)
path_data = master['pre']
path_data_small = small_master['pre']

In [3]:
path_data_small[0,0,0]

array([0., 0., 0.])

In [4]:
mapRand = path_data[random.randint(0,9999)]
mapRand.shape[0]

72

In [37]:
class TableEnv(Env):
    def __init__(self, map_bag = None):
        self.size = np.array([128,64])
        self.action_space = Discrete(4)
        self.observation_space = Dict({
            "agent": Box(low=np.array([0,0,0]), high = np.append(self.size,1), shape = (3,)),
            "target": Box(low=np.array([0,0]), high = self.size, shape = (2,)),
            #"field": Box(low=np.tile(np.array([0]),(72,128)), high = np.tile(np.array([255]),(72,128)), shape = (72,128)),
        })
        self.action_to_movement = {
            0: np.array([0,1,0]),
            1: np.array([1,0,0]),
            2: np.array([0,-1,0]),
            3: np.array([-1,0,0]),
            4: np.array([0,0,1])
        }
        self.agent_state = np.array([0,0,0])
        self.target = np.array([-1,-1])
        self.length = 200
        if map_bag.ndim == 4:
            self.map_bag = map_bag.mean(axis=3)
        else: 
            self.map_bag = map_bag
        self.numpy_map = self.map_bag[random.randint(0,self.map_bag.shape[0]-1)]
        self.obstacle_list, self.empty_list = self._obstacle_extraction(self.numpy_map)
        self._init_pygame()
    def _init_pygame(self):
        pygame.init()
        pygame.display.init()
        self.cell_size = 8
        self.offset = 1
        self.window_size = self.size*self.cell_size
        self.window_surface = pygame.display.set_mode(self.window_size)
        radius = self.cell_size/2 - self.offset
        
        self.agent_img = pygame.Surface((radius * 2, radius * 2), pygame.SRCALPHA)
        pygame.draw.circle(self.agent_img, (0,0,255), (radius,radius), radius) 
        
        self.tile_img = pygame.Surface((self.cell_size,self.cell_size),pygame.SRCALPHA)
        pygame.draw.rect(self.tile_img, (255,255,255),(0,0,self.cell_size,self.cell_size))
        pygame.draw.rect(self.tile_img, (128,128,128),(0,0,self.cell_size,self.cell_size), width=1)
        
        self.target_img = pygame.Surface((radius * 2, radius * 2), pygame.SRCALPHA)
        pygame.draw.circle(self.target_img, (0,255,0), (radius,radius), radius)
        
        self.obstacle_img = pygame.Surface((self.cell_size,self.cell_size), pygame.SRCALPHA)
        pygame.draw.rect(self.obstacle_img, (255,0,0), (0,0,self.cell_size,self.cell_size))
                         
        self.clock = pygame.time.Clock()
    def _random_location(self, allowed_pos, agent = False):
        point = random.choice(allowed_pos)
        if(agent):
            return np.array(np.append(point,[0]))
        else:
            return np.array(point)
    def _get_obs(self):
        return {"agent": self.agent_state, "target": self.target,} #"field": self.numpy_map}
    def _get_info(self):
        return {
            "distance": np.linalg.norm(
                self.agent_state - np.append(self.target,0), ord=1
            )
        }
    def _obstacle_extraction(self, numpy_map):
        point_list = []
        point_list2 = []
        debug_flag = 0
        x_axis = numpy_map.shape[1]
        y_axis = numpy_map.shape[0]
        for y in range(y_axis):
            for x in range(x_axis):
                if (numpy_map[y][x] >= 0.75):
                    point_list.append([x,y])
                else:
                    point_list2.append([x,y])
        #intersection = []
        #for point in point_list:
            #for point2 in point_list2:
               # if point == point2:
                   # intersection.append(point1)
        #print(np.array(intersection).shape)
        return np.array(point_list), np.array(point_list2)
    def step(self, action):
        terminated = False
        truncated = False
        reward = 0
        direction = self.action_to_movement[action]
        #print(self.agent_state+direction)
        self.agent_state = np.clip(
            self.agent_state+direction, 0, np.append(self.size - 1,2)
        )
        #if [self.agent_state[0], self.agent_state[1]] in self.obstacle_list:
            #reward -= 1
        distance = self._get_info()["distance"]
        reward -= .01*distance
        self.length -= 1
        if np.array_equal(self.agent_state, np.append(self.target, 0)):
            reward += 50
            terminated = True
            if self.agent_state[2] == 1:
                #reward += 80
                print("no")
        #if distance <= 80:
            #reward += 0.01
        if self.length <= 0:
            truncated = True
        observation = self._get_obs()
        info = self._get_info()
        return observation, reward, terminated, truncated, info
    def render(self):
        pygame.init()
        for r in range(self.size[1]):
            for c in range(self.size[0]):
                pos = [c*self.cell_size, r*self.cell_size]
                self.window_surface.blit(self.tile_img, pos)
                offset_pos = [pos[0]+self.offset,pos[1]+self.offset]
                if(np.array_equal([c,r,0], self.agent_state)):
                    self.window_surface.blit(self.agent_img, offset_pos)
                if(np.array_equal([c,r],self.target)):
                    self.window_surface.blit(self.target_img, offset_pos)
        #print(self.obstacle_list)
        for point in self.obstacle_list:
            point = [point[0]*self.cell_size, point[1]*self.cell_size]
            self.window_surface.blit(self.obstacle_img, point)
        pygame.display.update()
        pygame.event.pump()
        self.clock.tick(2)
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self.length = 200

        self.agent_state = self._random_location(self.empty_list, agent=True)
        self.target = self._random_location(self.empty_list)
        self.numpy_map = self.map_bag[random.randint(0,self.map_bag.shape[0]-1)]
        self.obstacle_list, self.empty_list = self._obstacle_extraction(self.numpy_map)
        checker = np.isin(self.obstacle_list, self.empty_list)

        observation = self._get_obs()
        info = self._get_info()
        return observation, info
    def close(self):
        pygame.quit()

In [8]:
def compare_2D(array1, array2):
    a1_view = array1.view([('', array1.dtype)] * array1.shape[1])
    a2_view = array2.view([('', array2.dtype)] * array2.shape[1])
    intersection = np.isin(a1_view,a2_view)
    return intersection.any()

a = np.array([2,4])
b = np.array([[1,2],[2,3],[3,4]])

#compare_2D(a,b)
np.expand_dims(a,0)

array([[2, 4]])

In [34]:
class TableEnvV1(Env):
    def __init__(self, map_bag = None):
        self.size = np.array([32,18])
        self.action_space = Discrete(4)
        self.window_size = 7
        self.pad_amount = int((self.window_size-1)/2)
        self.observation_space = Dict({
            "window": Box(low=0, high = 1, shape = (self.window_size,self.window_size)),
            #"target_info": Box(low = -1.0, high = 1.0, shape=(2,), dtype=np.float32),
            "agent": Box(low = 0, high = self.size, shape=(2,)),
            "target": Box(low=np.array([0,0]), high = self.size, shape = (2,))
        })
        self.action_to_movement = {
            0: np.array([0,1]),
            1: np.array([1,0]),
            2: np.array([0,-1]),
            3: np.array([-1,0]),
            4: np.array([0,0])
        }
        self.agent = np.array([-1,-1])
        self.target = np.array([-1,-1])
        self.length = 300
        self.max_repeats = 2
        if map_bag.ndim == 4:
            self.map_bag = map_bag.mean(axis=3)
        else: 
            self.map_bag = map_bag
        
        self.numpy_map = self.map_bag[random.randint(0,self.map_bag.shape[0]-1)]
        self.numpy_map = self._map_normalization(self.numpy_map)
        self.obstacle_list, self.empty_list = self._obstacle_extraction(self.numpy_map)
        self._init_pygame()
    def _init_pygame(self):
        pygame.init()
        pygame.display.init()
        self.cell_size = 32
        self.offset = 1
        self.window_size = self.size*self.cell_size
        self.window_surface = pygame.display.set_mode(self.window_size)
        radius = self.cell_size/2 - self.offset
        
        self.agent_img = pygame.Surface((radius * 2, radius * 2), pygame.SRCALPHA)
        pygame.draw.circle(self.agent_img, (0,0,255), (radius,radius), radius) 
        
        self.tile_img = pygame.Surface((self.cell_size,self.cell_size),pygame.SRCALPHA)
        pygame.draw.rect(self.tile_img, (255,255,255),(0,0,self.cell_size,self.cell_size))
        pygame.draw.rect(self.tile_img, (200,200,200),(0,0,self.cell_size,self.cell_size), width=1)
        
        self.target_img = pygame.Surface((radius * 2, radius * 2), pygame.SRCALPHA)
        pygame.draw.circle(self.target_img, (255,0,255), (radius,radius), radius)
        
        self.obstacle_img = pygame.Surface((self.cell_size,self.cell_size), pygame.SRCALPHA)
        pygame.draw.rect(self.obstacle_img, (255,0,0), (0,0,self.cell_size,self.cell_size))
                         
        self.clock = pygame.time.Clock()
    def _random_location(self, allowed_pos, agent = False):
        point = random.choice(allowed_pos)
        if(agent):
            return np.array(point)
        else:
            return np.array(point)
    def _get_obs(self):
        agent_pd = self.agent + self.pad_amount
        ax, ay = agent_pd[0], agent_pd[1]
        window = self.numpy_map_pd[ay - self.pad_amount:ay + self.pad_amount + 1,ax - self.pad_amount:ax + self.pad_amount + 1]
        return {"window": window, #"target_info":self._get_vec(),
                "agent": self.agent,
                "target":self.target,}
    def _get_distance(self, point1,point2):
        return np.linalg.norm(point1 - point2, ord=1)
    def _get_vec(self):
        vec = np.array([(self.target[0]-self.agent[0])/self.size[0],
                        (self.target[1]-self.agent[1])/self.size[1]])
        return vec
    def _get_info(self):
        return {"dummy": "dummy"}
    def _obstacle_extraction(self, numpy_map):
        point_list = []
        point_list2 = []
        debug_flag = 0
        x_axis = numpy_map.shape[1]
        y_axis = numpy_map.shape[0]
        for y in range(y_axis):
            for x in range(x_axis):
                if (numpy_map[y][x] >= 0.75):
                    point_list.append([x,y])
                else:
                    point_list2.append([x,y])
        #intersection = []
        #for point in point_list:
            #for point2 in point_list2:
               # if point == point2:
                   # intersection.append(point1)
        #print(np.array(intersection).shape)
        return np.array(point_list), np.array(point_list2)
    def _map_normalization(self, canvas):
        canvas = canvas.astype(np.float32)/255.0
        obstacles = (canvas>0.75)
        return obstacles
    def step(self, action):
        terminated = False
        truncated = False
        reward = 0
        direction = self.action_to_movement[action]
        self.agent = np.clip(
            self.agent+direction, 0, self.size - 1
        )
        x, y = self.agent
        self.visit_count[x,y] += 1
        distance = self._get_distance(self.agent, self.target)
        if self.visit_count[x,y] <= self.max_repeats:
            reward += -.1 
            #terminated = True
            #pass
        if compare_2D(np.expand_dims(self.agent,0), self.obstacle_list):
            reward += -1
            #print("caught")
        #reward += np.dot(direction, self._get_vec())/10 - 0.03
        #print(reward) #--------------------------------------------------------------------------------
        reward += -.1 * (distance - self.last_dis)
        #reward -= 0.025
        self.last_dis = distance
        self.length -= 1
        if np.array_equal(self.agent, self.target):
            reward += 50
            terminated = True
        #print(reward) #------------------------------------------------------------------------------------------------------------
        if self.length <= 0:
            truncated = True
        observation = self._get_obs()
        info = self._get_info()
        return observation, reward, terminated, truncated, info
    def render(self):
        pygame.init()
        #print(self.agent)
        for r in range(self.size[1]):
            for c in range(self.size[0]):
                pos = [c*self.cell_size, r*self.cell_size]
                self.window_surface.blit(self.tile_img, pos)
                offset_pos = [pos[0]+self.offset,pos[1]+self.offset]
                if(np.array_equal([c,r], self.agent)):
                    self.window_surface.blit(self.agent_img, offset_pos)
                if(np.array_equal([c,r],self.target)):
                    self.window_surface.blit(self.target_img, offset_pos)
        #print(self.obstacle_list)
        for point in self.obstacle_list:
            point = [point[0]*self.cell_size, point[1]*self.cell_size]
            self.window_surface.blit(self.obstacle_img, point)
            if (point == self.agent).all():
                self.window_surface.blit(self.agent_img, point)
            if (point == self.target).all():
                self.window_surface.blit(self.target_img, point)
        pygame.display.update()
        pygame.event.pump()
        self.clock.tick(5)
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self.length = 300
        self.numpy_map = self.map_bag[random.randint(0,self.map_bag.shape[0]-1)]
        self.numpy_map = self._map_normalization(self.numpy_map)
        self.numpy_map_pd = np.pad(self.numpy_map, pad_width=((self.pad_amount,self.pad_amount),(self.pad_amount,self.pad_amount)), mode = "constant", constant_values=1)
        self.obstacle_list, self.empty_list = self._obstacle_extraction(self.numpy_map)
        #print(self.obstacle_list)
        self.agent = self._random_location(self.empty_list)
        self.target = self._random_location(self.empty_list)
        self.visit_count = np.zeros((self.size[0], self.size[1]))
        self.last_dis = self._get_distance(self.agent, self.target)
        #print(self.target)
        #print(self.agent, "a")
        if compare_2D(np.expand_dims(self.target,0), self.obstacle_list):
            print(self.target, "fail")
        observation = self._get_obs()
        info = self._get_info()
        return observation, info
    def close(self):
        pygame.quit()

In [38]:
env = TableEnv(path_data)
#env.numpy_map

In [33]:
episodes = 10
for episode in range(1, 1+episodes):
    state = env.reset()

    done = False
    score = 0

    while not done:
        env.render()
        action = env.action_space.sample()
        n_state, reward, terminated, truncated, info,  = env.step(action)
        score += float(reward)
        if terminated or truncated:
            #print(env.length)
            done = True
    print(f"Episode: {episode} Score: {score}")

KeyboardInterrupt: 

In [73]:
env.step(0)
print("")

0.44166666666666665



In [113]:
#env.close()
env.reset()
while True:
    rand_choice = random.randint(0,3)
    env.step(rand_choice)
    env.render()

caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught
caught


KeyboardInterrupt: 

In [39]:
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
import os
#env = DummyVecEnv([lambda: env])
#env = Monitor(env)
log_path = os.path.join('./', "Logs")

model = PPO("MultiInputPolicy",  env, verbose=1, tensorboard_log = log_path)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [42]:
model.learn(total_timesteps = 200000)

Logging to ./Logs\PPO_143
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 75.7     |
|    ep_rew_mean     | 23.5     |
| time/              |          |
|    fps             | 2349     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 85.2       |
|    ep_rew_mean          | 16.9       |
| time/                   |            |
|    fps                  | 1577       |
|    iterations           | 2          |
|    time_elapsed         | 2          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.01057532 |
|    clip_fraction        | 0.13       |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.688     |
|    explained_variance   | 0.958      |
|    learning_rate 

In [13]:
episodes = 10
for episode in range(episodes):
    obs, info1 = env.reset()
    print(env.target)
    done = False
    score = 0
    while not done:
        env.render()
        action, info1 = model.predict(obs)
        #print(action.item)
        obs, reward, terminated, truncated, info = env.step(float(action))
        score += reward
        if terminated or truncated:
            done = True
    
    print(f"Episode: {episode}, Score{score}")

[13 32]
Episode: 0, Score21.22
[95 16]


KeyboardInterrupt: 

In [157]:
Model_Path = os.path.join("Models","PPO_OBS")
model.save(Model_Path)